# High-level developer example: scVI on Dataset 2

This notebook uses the generic **MethodSpec → benchmark_method** API. scVI remains user-owned example code; scRareBench does not implement or register scVI internally.

Change `METHOD_SEEDS` to one integer for a quick run or several seeds for a reproducible multi-seed benchmark. Method dependencies, method preprocessing/training, and method configuration are all controlled in this notebook.


In [ ]:
# Optional diagnostic only: leave commented unless you want to inspect the runtime.
# import sys
# from importlib import metadata
# print(sys.version)
# for pkg in ("numpy", "torch", "jax", "scvi-tools"):
#     try: print(pkg, metadata.version(pkg))
#     except metadata.PackageNotFoundError: print(pkg, "not installed")


In [ ]:
import subprocess, sys
REPO = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@v0.10.6"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", REPO])

# Method dependencies are explicitly user-controlled.
METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)
INSTALL_METHOD_DEPENDENCIES = True

from scrarebench.runtime import setup_runtime
setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES if INSTALL_METHOD_DEPENDENCIES else (),
    extra_imports=("scvi",) if INSTALL_METHOD_DEPENDENCIES else (),
    quiet=False,
)


In [ ]:
from scrarebench import load_dataset, dataset_info
adata = load_dataset(2)
info = dataset_info(adata)
print(adata)
print(info)


In [ ]:
# Dataset 2 count/HVG contract: recover a validated raw-count layer before method copies are made.
import numpy as np
import pandas as pd
import scipy.sparse as sp

def sampled_count_diagnostics(matrix, max_values=100_000, seed=42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    if len(values) > max_values:
        rng = np.random.default_rng(seed)
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    return {
        "finite_fraction": float(np.mean(np.isfinite(values))) if len(values) else 1.0,
        "minimum": float(np.min(finite)) if len(finite) else 0.0,
        "nonnegative": bool(len(finite) == 0 or finite.min() >= 0),
        "integer_like": bool(len(finite) == 0 or np.allclose(finite, np.rint(finite), atol=1e-6)),
    }

def _copy_matrix(matrix):
    return matrix.copy().tocsr() if sp.issparse(matrix) else np.asarray(matrix).copy()

def ensure_counts_layer(adata, layer_key="counts"):
    if layer_key in adata.layers:
        diag = sampled_count_diagnostics(adata.layers[layer_key])
        if diag["nonnegative"] and diag["integer_like"]:
            return f"layers/{layer_key}", diag
        raise ValueError(f"Existing {layer_key!r} layer is not nonnegative integer-like raw counts: {diag}")

    if adata.raw is not None:
        raw_names = pd.Index(adata.raw.var_names.astype(str))
        current_names = pd.Index(adata.var_names.astype(str))
        positions = raw_names.get_indexer(current_names)
        if np.all(positions >= 0):
            matrix = adata.raw.X[:, positions]
            diag = sampled_count_diagnostics(matrix)
            if diag["nonnegative"] and diag["integer_like"]:
                adata.layers[layer_key] = _copy_matrix(matrix)
                return "raw.X aligned to adata.var_names", diag

    diag = sampled_count_diagnostics(adata.X)
    if diag["nonnegative"] and diag["integer_like"]:
        adata.layers[layer_key] = _copy_matrix(adata.X)
        return "X", diag

    raise ValueError(
        "Could not locate nonnegative integer-like raw counts for Dataset 2. "
        f"layers={list(adata.layers)}, raw_present={adata.raw is not None}, X_diagnostics={diag}"
    )

if info.get("scib_hvg_batch_mode") != "global":
    raise RuntimeError(f"Dataset 2 is expected to use global HVG selection, observed metadata: {info}")
count_layer = info.get("count_layer") or "counts"
count_source, count_diagnostics = ensure_counts_layer(adata, count_layer)
print("Raw-count source:", count_source)
print("Count diagnostics:", count_diagnostics)


In [ ]:
# Reproducibility controls: method seeds vary; benchmark seed stays fixed.
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a quick single-seed run
BENCHMARK_SEED = 42

# Dataset 2 uses the validated global Seurat-v3 HVG policy. The combined
# donor_id × assay evaluation batch is intentionally NOT passed to HVG selection.
METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_policy": "global_seurat_v3_raw_counts",
    "hvg_batch_key": None,
    "hvg_span": 0.3,
    "n_hidden": 128,
    "n_latent": 30,
    "n_layers": 2,
    "dropout_rate": 0.10,
    "dispersion": "gene-batch",
    "gene_likelihood": "nb",
    "max_epochs": 200,
    "batch_size": 256,
}


## User-owned method section

This function is the only method-specific part. Replace it with any batch-effect-removal/integration method. The contract is simply one latent row per benchmark cell. Dependencies and configuration can change without modifying scRareBench.


In [ ]:
import numpy as np
import scanpy as sc
import scvi
from scrarebench import MethodOutput

def run_scvi(method_adata, seed, config):
    batch_key = info["batch_key"]
    count_layer = info.get("count_layer") or "counts"
    if count_layer not in method_adata.layers:
        raise KeyError(
            f"Validated raw-count layer {count_layer!r} is missing from the method copy. "
            "Run the Dataset 2 count-preparation cell before benchmark_method()."
        )

    hvg_kwargs = {
        "layer": count_layer,
        "flavor": "seurat_v3",
        "n_top_genes": min(int(config["n_hvg"]), method_adata.n_vars),
        "span": float(config["hvg_span"]),
        "subset": False,
        "check_values": True,
    }
    if config.get("hvg_batch_key") is not None:
        hvg_kwargs["batch_key"] = config["hvg_batch_key"]
    sc.pp.highly_variable_genes(method_adata, **hvg_kwargs)

    hvg_mask = method_adata.var["highly_variable"].fillna(False).to_numpy()
    expected_hvg = min(int(config["n_hvg"]), method_adata.n_vars)
    if int(hvg_mask.sum()) != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} HVGs under the fixed Dataset 2 policy but selected {int(hvg_mask.sum())}."
        )
    method_adata = method_adata[:, hvg_mask].copy()

    scvi.settings.seed = int(seed)
    scvi.model.SCVI.setup_anndata(method_adata, layer=count_layer, batch_key=batch_key)
    model = scvi.model.SCVI(
        method_adata,
        n_hidden=config["n_hidden"],
        n_latent=config["n_latent"],
        n_layers=config["n_layers"],
        dropout_rate=config["dropout_rate"],
        dispersion=config["dispersion"],
        gene_likelihood=config["gene_likelihood"],
    )
    model.train(
        max_epochs=config["max_epochs"],
        train_size=0.90,
        validation_size=0.10,
        batch_size=config["batch_size"],
        early_stopping=True,
        early_stopping_patience=20,
        accelerator="auto",
        devices="auto",
    )
    latent = model.get_latent_representation()
    return MethodOutput(latent=latent, barcodes=method_adata.obs_names)


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scVI",
    runner=run_scvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    benchmark_config={"random_state": BENCHMARK_SEED},
    # Dependencies were handled above with setup_runtime; keep this False.
    install_dependencies=False,
    finalize=True,
)


In [ ]:
from IPython.display import display
display(result.summary().round(5))
print("Seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)
